# Ciringoni M4 uji akhir konfigurasi terkunci

Jalankan satu inferensi test pada 178 gambar dan 1.159 anotasi. Commit GitHub dan hash lock harus cocok sebelum memproses test.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = "https://github.com/Jordyee/ciringoni-data-mining.git"
SOURCE = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
PROJECT = Path("/tmp/ciringoni-source/Data_Mining_Ciringoni_M1")
SCRIPTS = PROJECT / "hasil_m2"

def execute(*args):
    subprocess.run([str(arg) for arg in args], cwd=PROJECT, check=True)

def input_file(name):
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f"Expected one mounted {name}, found {len(paths)}"
    return paths[0]

subprocess.run(["git", "clone", "--quiet", REPO, str(Path("/tmp/ciringoni-source"))], check=True)
execute("git", "checkout", "--quiet", SOURCE)


## Lingkungan inferensi


In [ ]:
execute(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210", "pycocotools==2.0.11", "openvino==2026.4.1", "nncf==3.4.0")


## Verifikasi lock dan jalankan test


In [ ]:
from tarfile import open as tar_open
from urllib.request import urlopen
import csv
import re

LOCK_COMMIT = "010200666e540a7b425b6be28c7bcb328b8d0191"
LOCK_PATH = "kaggle/evidence/m4-e7/test-lock-kaggle.json"
LOCK_SHA256 = "96ad00cedcf40a62749bfe4fcf5e27f39509df9651e2702efdfb6cd68bbdb2e0"
assert re.fullmatch("[0-9a-f]{40}", LOCK_COMMIT), "Use the full development commit already containing the lock"
assert re.fullmatch("[0-9a-f]{64}", LOCK_SHA256)
lock_url = f"https://raw.githubusercontent.com/Jordyee/ciringoni-data-mining/{LOCK_COMMIT}/{LOCK_PATH}"
with urlopen(lock_url, timeout=60) as response:
    lock_raw = response.read()
assert hashlib.sha256(lock_raw).hexdigest() == LOCK_SHA256
assert input_file("test-lock-kaggle.json").read_bytes() == lock_raw
lock = json.loads(lock_raw)
output = WORK / "m4-test"
output.mkdir()
lock_local = output / "test-lock-kaggle.json"
lock_local.write_bytes(lock_raw)
print("GitHub pre-test lock commit:", LOCK_COMMIT, "| SHA-256:", LOCK_SHA256)

archives = list(INPUT.rglob("test-native.tar"))
assert len(archives) <= 1
if archives:
    archive = archives[0]
    with archive.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == "a414efb658ad3d6d2b7172bcaa79f2dbb39818851b88748f11d2cc67864f792a"
    test_input = Path("/tmp/ciringoni-native-test")
    test_input.mkdir()
    with tar_open(archive) as packed:
        packed.extractall(test_input, filter="data")
else:
    test_input = input_file("audit_test_native.json").parent
    assert (test_input / "coco_test_gt.json").is_file()
    assert all((test_input / str(size) / "images/test").is_dir() for size in (640, 1280))
for size in (640, 1280):
    assert not (test_input / str(size) / "images/train").exists()
    assert not (test_input / str(size) / "images/val").exists()

export = input_file("comparison.json").parent
precision = lock["precision"]
checkpoint = export / "candidate.pt"
validation_metrics = export / precision / "metrics.json"
models = []
for candidate in export.rglob("*.xml"):
    with candidate.open("rb") as stream:
        if lock["model_sha256"].get(candidate.name) == hashlib.file_digest(stream, "sha256").hexdigest():
            models.append(candidate.parent)
assert len(models) == 1
params = [sys.executable, SCRIPTS / "evaluasi_final.py", "test", models[0], test_input, output / "evaluation",
          "--size", lock["size"], "--precision", precision, "--checkpoint", checkpoint,
          "--validation-metrics", validation_metrics, "--lock", lock_local]
execute(*params)

metrics = json.loads((output / "evaluation/metrics.json").read_text())
latency = json.loads((output / "evaluation/latency.json").read_text())
assert len(latency) == metrics["dataset"]["images"] == 178 and metrics["dataset"]["boxes"] == 1159
assert len({row["image_id"] for row in latency}) == 178
assert metrics["model_sha256"] == lock["model_sha256"]
row = {"id": f"OpenVINO_{precision}", "split": "test", "images": 178, "boxes": 1159, "github_lock_commit": LOCK_COMMIT,
       **{key: metrics[key] for key in ("size", "ap_small", "ap_50_95", "ap_50", "precision", "recall", "cpu_median_ms", "cpu_p95_ms")}}
with (output / "eksperimen.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)
print(json.dumps(row, indent=2))
